In [ ]:
import pandas as pd 
    
    # Load time series data from CSV file
file_path = "/Users/igormol/Desktop/John/0105010006.csv"
data = pd.read_csv(file_path)

In [ ]:
import pandas as pd

# Assuming your original data frame is named "data"

# Create a new_data data frame with TIME_SCALE ranging from 1 to 135
new_data = pd.DataFrame({'TIME_SCALE': range(1, 136)})

# Merge new_data with the original data on TIME_SCALE using left join
new_data = new_data.merge(data, on='TIME_SCALE', how='left')

# Fill NaN values in COUNT column with 0
new_data['COUNT'].fillna(0, inplace=True)

# If you want to ensure COUNT column is of integer type, you can convert it
new_data['COUNT'] = new_data['COUNT'].astype(int)

# Display the resulting new_data data frame
print(new_data)
data = new_data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Function to detect and substitute outliers using Z-score
def z_score_outlier_detection(data_frame, column, threshold=3):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data
def plot_original_vs_cleaned(original_data, cleaned_data):
    plt.figure(figsize=(10, 6))
    plt.plot(original_data['TIME_SCALE'], original_data['COUNT'], label='Original Data', marker='o')
    plt.plot(cleaned_data['TIME_SCALE'], cleaned_data['COUNT'], label='Cleaned Data', marker='X')
    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = z_score_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the original and cleaned data
original_data = data.copy()
cleaned_data = save_cleaned_data(original_data, 'COUNT')
plot_original_vs_cleaned(original_data, cleaned_data)

# Save the cleaned data in a new_data data frame
new_data = cleaned_data.copy()

# Display the resulting time series
print("Original Data:")
print(original_data)
print("\nCleaned Data:")
print(new_data)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=2, months_to_predict=12, randomness_factor=0.1, smoothing_factor=1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.smoothing_factor = smoothing_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def exponential_smoothing(self, series):
        smoothed_series = [series[0]]  # Assuming the first value remains the same

        for i in range(1, len(series)):
            smoothed_value = self.smoothing_factor * series[i] + (1 - self.smoothing_factor) * smoothed_series[-1]
            smoothed_series.append(smoothed_value)

        return smoothed_series

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            # Apply exponential smoothing to the current window
            y_smoothed = self.exponential_smoothing(y)

            model = self.train_model(X, y_smoothed)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)  # Adjust the smoothing factor as needed
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class SeasonalityDetector:
    def __init__(self, data, window_size=2, smoothing_factor=0.2):
        self.data = data
        self.window_size = window_size
        self.smoothing_factor = smoothing_factor
        self.detected_seasonality = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def exponential_smoothing(self, series):
        smoothed_series = [series[0]]  # Assuming the first value remains the same

        for i in range(1, len(series)):
            smoothed_value = self.smoothing_factor * series[i] + (1 - self.smoothing_factor) * smoothed_series[-1]
            smoothed_series.append(smoothed_value)

        return smoothed_series

    def detect_seasonality(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            # Apply exponential smoothing to the current window
            y_smoothed = self.exponential_smoothing(y)

            seasonal_pattern = np.mean(y - y_smoothed)
            self.detected_seasonality.append(seasonal_pattern)

    def print_seasonality_table(self):
        results_table = pd.DataFrame({
            'Window Index': range(len(self.detected_seasonality)),
            'Detected Seasonality': self.detected_seasonality
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

    def plot_seasonality(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        plt.plot(self.detected_seasonality, label='Detected Seasonality', linestyle='dashed', color='red')

        plt.xlabel('Window Index')
        plt.ylabel('Detected Seasonality')
        plt.title('Seasonality Detection in Time Series')
        plt.legend()
        plt.show()

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    
    seasonality_detector = SeasonalityDetector(data)
    seasonality_detector.preprocess_data()
    seasonality_detector.detect_seasonality()
    seasonality_detector.print_seasonality_table()
    seasonality_detector.plot_seasonality()

if __name__ == "__main__":
    main()
